### Outlier Detection & Handling

Loading cleaned data

In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
df = pd.read_csv("../outputs/cleaned.csv")
print("Loaded shape:", df.shape)

features = [
    "RevolvingUtilization", "Age", "DebtRatio", "MonthlyIncome",
    "OpenCreditLines", "RealEstateLines", "Dependents",
    "Times30_59Late", "Times60_89Late", "Times90Late"
]

Loaded shape: (149233, 11)


IQR Method to Detect Outliers

In [3]:
print("Outlier Detection (IQR Method):")
print(f"  {'Feature':<28} {'Below Q1-1.5IQR':>15} {'Above Q3+1.5IQR':>15} {'Total':>8}")
print("  " + "-" * 70)

for col in features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = (df[col] < Q1 - 1.5 * IQR).sum()
    upper = (df[col] > Q3 + 1.5 * IQR).sum()
    print(f"  {col:<28} {lower:>15,} {upper:>15,} {lower + upper:>8,}")

Outlier Detection (IQR Method):
  Feature                      Below Q1-1.5IQR Above Q3+1.5IQR    Total
  ----------------------------------------------------------------------
  RevolvingUtilization                       0             769      769
  Age                                        0              44       44
  DebtRatio                                  0          31,276   31,276
  MonthlyIncome                              0           9,093    9,093
  OpenCreditLines                            0           3,980    3,980
  RealEstateLines                            0             793      793
  Dependents                                 0          13,336   13,336
  Times30_59Late                             0          23,928   23,928
  Times60_89Late                             0           7,550    7,550
  Times90Late                                0           8,272    8,272


In [4]:
# Late payment columns have sentinel codes 96 and 98
for col in ["Times30_59Late", "Times60_89Late", "Times90Late"]:
    count = (df[col] >= 96).sum()
    print(f"  {col}: {count} rows with value >= 96 (sentinel codes, not real counts)")

# RevolvingUtilization should be 0-1 normally
extreme = (df["RevolvingUtilization"] > 1).sum()
print(f"  RevolvingUtilization: {extreme:,} rows > 1 (exceeds credit limit or data error)")

extreme_dr = (df["DebtRatio"] > 10).sum()
print(f"  DebtRatio: {extreme_dr:,} rows > 10 (likely data errors)")

# MonthlyIncome = 0
zero_inc = (df["MonthlyIncome"] == 0).sum()
print(f"  MonthlyIncome: {zero_inc:,} rows = 0")

# Min/Max for each feature 
print(f"\nCurrent Min/Max values:")
print(f"  {'Feature':<28} {'Min':>12} {'Max':>12}")
print("  " + "-" * 55)
for col in features:
    print(f"  {col:<28} {df[col].min():>12.2f} {df[col].max():>12.2f}")

  Times30_59Late: 216 rows with value >= 96 (sentinel codes, not real counts)
  Times60_89Late: 216 rows with value >= 96 (sentinel codes, not real counts)
  Times90Late: 216 rows with value >= 96 (sentinel codes, not real counts)
  RevolvingUtilization: 3,321 rows > 1 (exceeds credit limit or data error)
  DebtRatio: 28,871 rows > 10 (likely data errors)
  MonthlyIncome: 1,616 rows = 0

Current Min/Max values:
  Feature                               Min          Max
  -------------------------------------------------------
  RevolvingUtilization                 0.00     50708.00
  Age                                 21.00       109.00
  DebtRatio                            0.00    329664.00
  MonthlyIncome                        0.00   3008750.00
  OpenCreditLines                      0.00        58.00
  RealEstateLines                      0.00        54.00
  Dependents                           0.00        20.00
  Times30_59Late                       0.00        98.00
  Times60_89La

Handling outliers

In [5]:
# 1. Cap sentinel values in late payment columns 
# Values 96 and 98 are codes, not real counts. Cap at 20.

late_cols = ["Times30_59Late", "Times60_89Late", "Times90Late"]
print("1. Late payment sentinel values (96, 98):")
for col in late_cols:
    count = (df[col] >= 96).sum()
    print(f"   {col}: {count} rows >= 96 -> capped at 20")
    df.loc[df[col] >= 96, col] = 20

# Store cap values to reuse on test set
cap_values = {}

# 2. Cap RevolvingUtilization at 99th percentile 
cap = df["RevolvingUtilization"].quantile(0.99)
count = (df["RevolvingUtilization"] > cap).sum()
df["RevolvingUtilization"] = df["RevolvingUtilization"].clip(upper=cap)
cap_values["RevolvingUtilization"] = cap
print(f"\n2. RevolvingUtilization: {count:,} rows capped at p99 ({cap:.4f})")

# 3. Cap DebtRatio at 99th percentile 
cap = df["DebtRatio"].quantile(0.99)
count = (df["DebtRatio"] > cap).sum()
df["DebtRatio"] = df["DebtRatio"].clip(upper=cap)
cap_values["DebtRatio"] = cap
print(f"3. DebtRatio: {count:,} rows capped at p99 ({cap:.2f})")

# 4. Cap MonthlyIncome at 99th percentile 
cap = df["MonthlyIncome"].quantile(0.99)
count = (df["MonthlyIncome"] > cap).sum()
df["MonthlyIncome"] = df["MonthlyIncome"].clip(upper=cap)
cap_values["MonthlyIncome"] = cap
print(f"4. MonthlyIncome: {count:,} rows capped at p99 ({cap:.0f})")

# 5. Cap OpenCreditLines at 99th percentile 
cap = df["OpenCreditLines"].quantile(0.99)
count = (df["OpenCreditLines"] > cap).sum()
df["OpenCreditLines"] = df["OpenCreditLines"].clip(upper=cap)
cap_values["OpenCreditLines"] = cap
print(f"5. OpenCreditLines: {count:,} rows capped at p99 ({cap:.0f})")

# 6. Cap RealEstateLines at 99th percentile 
cap = df["RealEstateLines"].quantile(0.99)
count = (df["RealEstateLines"] > cap).sum()
df["RealEstateLines"] = df["RealEstateLines"].clip(upper=cap)
cap_values["RealEstateLines"] = cap
print(f"6. RealEstateLines: {count:,} rows capped at p99 ({cap:.0f})")

1. Late payment sentinel values (96, 98):
   Times30_59Late: 216 rows >= 96 -> capped at 20
   Times60_89Late: 216 rows >= 96 -> capped at 20
   Times90Late: 216 rows >= 96 -> capped at 20

2. RevolvingUtilization: 1,492 rows capped at p99 (1.0940)
3. DebtRatio: 1,493 rows capped at p99 (4988.04)
4. MonthlyIncome: 1,493 rows capped at p99 (23098)
5. OpenCreditLines: 1,476 rows capped at p99 (24)
6. RealEstateLines: 1,482 rows capped at p99 (4)


In [6]:
print("After treatment:")
print(f"  {'Feature':<28} {'Min':>10} {'Max':>10} {'Median':>10}")
print("  " + "-" * 60)
for col in features:
    print(f"  {col:<28} {df[col].min():>10.2f} {df[col].max():>10.2f} {df[col].median():>10.2f}")

After treatment:
  Feature                             Min        Max     Median
  ------------------------------------------------------------
  RevolvingUtilization               0.00       1.09       0.15
  Age                               21.00     109.00      52.00
  DebtRatio                          0.00    4988.04       0.37
  MonthlyIncome                      0.00   23097.76    5400.00
  OpenCreditLines                    0.00      24.00       8.00
  RealEstateLines                    0.00       4.00       1.00
  Dependents                         0.00      20.00       0.00
  Times30_59Late                     0.00      20.00       0.00
  Times60_89Late                     0.00      20.00       0.00
  Times90Late                        0.00      20.00       0.00


Saving the csv file

In [8]:
df.to_csv("../outputs/outliers_handled.csv", index=False)
print(f"Saved: outputs/outliers_handled.csv ({df.shape[0]:,} rows)")

with open("../outputs/outlier_caps.pkl", "wb") as f:
    pickle.dump(cap_values, f)
print(f"Saved: outputs/outlier_caps.pkl")
print(f"Cap values: {cap_values}")

Saved: outputs/outliers_handled.csv (149,233 rows)
Saved: outputs/outlier_caps.pkl
Cap values: {'RevolvingUtilization': np.float64(1.093976506), 'DebtRatio': np.float64(4988.039999999979), 'MonthlyIncome': np.float64(23097.75999999995), 'OpenCreditLines': np.float64(24.0), 'RealEstateLines': np.float64(4.0)}
